In [14]:
import os
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterstats import zonal_stats

sys.path.insert(0, os.path.abspath("../.."))
from lib import io as lio

In [ ]:
pumas = gpd.read_file("../geometry/pumas/ipums_puma_2010.shp")
pumas = pumas[~pumas["STATEFIP"].isin([2, 15, 72])]
pumas

,GISMATCH,GISJOIN,GEOID,STATEFIP,State,PUMA,Name,geometry
0,600105,G06000105,0600105,06,California,00105,"Alameda County (West)--San Leandro, Alameda & ...","MULTIPOLYGON (((-2258646.53 342733.341, -22586..."
1,600102,G06000102,0600102,06,California,00102,Alameda County (Northwest)--Oakland (Northwest...,"POLYGON ((-2259099.372 353383.127, -2259101.40..."
2,608502,G06008502,0608502,06,California,08502,Santa Clara County (Northwest)--Sunnyvale & Sa...,"POLYGON ((-2246220.131 304622.39, -2246216.047..."
3,600108,G06000108,0600108,06,California,00108,"Alameda County (Southwest)--Union City, Newark...","MULTIPOLYGON (((-2247215.331 305414.315, -2247..."
4,600107,G06000107,0600107,06,California,00107,Alameda County (Central)--Hayward City PUMA,"POLYGON ((-2244205.948 331582.334, -2244209.13..."
...,...,...,...,...,...,...,...,...
2373,4700500,G47000500,4700500,47,Tennessee,00500,Sumner County--Hendersonville City PUMA,"POLYGON ((866114.245 -51743.712, 866113.249 -5..."
2374,4702501,G47002501,4702501,47,Tennessee,02501,Nashville-Davidson (East) PUMA,"MULTIPOLYGON (((843851.3 -110459.794, 843847.3..."
2375,5310400,G53010400,5310400,53,Washington,10400,"Stevens, Okanogan, Pend Oreille & Ferry Counti...","POLYGON ((-1820226.155 1520836.039, -1819540.5..."
2376,5310300,G53010300,5310300,53,Washington,10300,Chelan & Douglas Counties PUMA,"POLYGON ((-1833489.407 1472988.079, -1833401.9..."


In [3]:
with rasterio.open("jan_temp/prism_tmean_us_25m_202001_avg_30y.tif") as src:
    pumas = pumas.to_crs(src.crs)

In [ ]:
data = [
    ("jan_temp/prism_tmean_us_25m_202001_avg_30y.tif", "JAN_AVG_TEMP_C"),
    ("july_temp/prism_tmean_us_25m_202007_avg_30y.tif", "JULY_AVG_TEMP_C"),
    ("precipitation/prism_ppt_us_25m_2020_avg_30y.tif", "AVG_TOT_PPT_MM"),
]

In [5]:
for file, col in data:
    stats = zonal_stats(
        pumas,
        file,
        stats=["mean"],
        all_touched=True,
    )
    normalizer = 1
    pumas[col] = [s["mean"] for s in stats]

In [28]:
pumas["PUMA"] = pumas["GISMATCH"].astype(str).str.zfill(7)
pumas["AVG_TOT_PPT_M"] = pumas["AVG_TOT_PPT_MM"] / 1000
pumas = pumas.set_index("PUMA", drop=True).dropna()
pumas

,GISMATCH,GISJOIN,GEOID,STATEFIP,State,Name,geometry,JAN_AVG_TEMP_C,JULY_AVG_TEMP_C,AVG_TOT_PPT_MM,AVG_TOT_POP_M,AVG_TOT_PPT_M
PUMA,,,,,,,,,,,,
0600105,600105,G06000105,0600105,06,California,"Alameda County (West)--San Leandro, Alameda & ...","MULTIPOLYGON (((-122.21426 37.76078, -122.2136...",10.580900,17.592436,528.727155,0.528727,0.528727
0600102,600102,G06000102,0600102,06,California,Alameda County (Northwest)--Oakland (Northwest...,"POLYGON ((-122.25253 37.8511, -122.25252 37.85...",10.794944,17.426832,572.573134,0.572573,0.572573
0608502,608502,G06008502,0608502,06,California,Santa Clara County (Northwest)--Sunnyvale & Sa...,"POLYGON ((-121.95931 37.46352, -121.95926 37.4...",10.203128,20.149778,395.343192,0.395343,0.395343
0600108,600108,G06000108,0600108,06,California,"Alameda County (Southwest)--Union City, Newark...","MULTIPOLYGON (((-121.97269 37.46792, -121.9722...",10.050426,19.614747,420.711829,0.420712,0.420712
0600107,600107,G06000107,0600107,06,California,Alameda County (Central)--Hayward City PUMA,"POLYGON ((-122.02038 37.69999, -122.02032 37.6...",9.821756,19.351513,497.514606,0.497515,0.497515
...,...,...,...,...,...,...,...,...,...,...,...,...
4700500,4700500,G47000500,4700500,47,Tennessee,Sumner County--Hendersonville City PUMA,"POLYGON ((-86.20552 36.63954, -86.20557 36.639...",2.544301,25.794569,1347.555114,1.347555,1.347555
4702501,4702501,G47002501,4702501,47,Tennessee,Nashville-Davidson (East) PUMA,"MULTIPOLYGON (((-86.52158 36.13816, -86.52174 ...",3.213679,26.287416,1335.690814,1.335691,1.335691
5310400,5310400,G53010400,5310400,53,Washington,"Stevens, Okanogan, Pend Oreille & Ferry Counti...","POLYGON ((-120.8512 49.00036, -120.84163 49.00...",-3.743813,18.306368,632.406362,0.632406,0.632406


In [9]:
puma_pop = lio.read_puma_pop("../geometry/pumas/puma_pop_2013_2018.csv")
puma_pop.head()

,Geo_NAME,Geo_qname,Geo_STUSAB,Geo_SUMLEV,Geo_GEOCOMP,Geo_FILEID,Geo_LOGRECNO,Geo_US,Geo_REGION,Geo_DIVISION,...,Geo_SUBMCD,Geo_SDELM,Geo_SDSEC,Geo_SDUNI,Geo_UR,Geo_PCI,Geo_BTTR,Geo_BTBG,Geo_PUMA5,TOT_POP
PUMA,,,,,,,,,,,,,,,,,,,,,
0100100,Lauderdale,"Lauderdale, Colbert, Franklin & Marion (Northe...",AL,795,0,ACSSF,8665,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100,185958
0100200,Limestone & Madison (Outer) Counties--Huntsvil...,Limestone & Madison (Outer) Counties--Huntsvil...,AL,795,0,ACSSF,8666,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200,191502
0100301,Huntsville (North) & Madison (East) Cities PUMA,Huntsville (North) & Madison (East) Cities PUM...,AL,795,0,ACSSF,8667,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,301,126711
0100302,Huntsville City (Central & South) PUMA,"Huntsville City (Central & South) PUMA, Alabama",AL,795,0,ACSSF,8668,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,302,102033
0100400,DeKalb & Jackson Counties PUMA,"DeKalb & Jackson Counties PUMA, Alabama",AL,795,0,ACSSF,8669,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,400,123294


In [29]:
puma_migpuma = lio.load_puma_migpuma("../geometry/equivalencies/puma_migpuma_2010.csv")
puma_migpuma = puma_migpuma[~puma_migpuma["State"].astype(int).isin([2, 15, 72])]
puma_migpuma.head()

,State,MIGPUMA
PUMA,,
0100100,01,0100190
0100200,01,0100290
0100301,01,0100290
0100302,01,0100290
0100400,01,0100400


In [30]:
cols = ["JAN_AVG_TEMP_C", "JULY_AVG_TEMP_C", "AVG_TOT_PPT_M"]
puma_migpuma["TOT_POP"] = puma_pop.loc[puma_migpuma.index, "TOT_POP"]
puma_migpuma[cols] = pumas.loc[puma_migpuma.index, cols]

In [ ]:
def wavg(g, cols, w="TOT_POP"):
    return pd.Series({c: np.average(g[c], weights=g[w]) for c in cols})


migpumas = (
    puma_migpuma.groupby("MIGPUMA")
    .apply(wavg, cols=cols, include_groups=False)
    .reset_index()
    .set_index("MIGPUMA")
    .dropna()
)

/tmp/ipykernel_781117/3809598267.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  migpumas = puma_migpuma.groupby("MIGPUMA").apply(wavg, cols=cols).reset_index()


In [33]:
pumas.drop(columns=["geometry"]).to_csv("puma_weather.csv")
migpumas.to_csv("migpuma_weather.csv")